install the current SDK

In [1]:
!pip install -q google-genai      # the NEW unified SDK. NOT 'google-generativeai'

Connect To drive and set up client

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
from google import genai
from google.genai import types
from google.colab import userdata

PROJECT_ROOT = '/content/drive/MyDrive/Projects/multi-agent-discovery'
os.makedirs(os.path.join(PROJECT_ROOT, 'src/llm'), exist_ok=True)

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))   # key from Colab Secrets (Day 1)
print("client ready")

Mounted at /content/drive
client ready


list the models your key can call

In [ ]:
for m in client.models.list():
    print(" ", m.name)

  models/gemini-2.5-flash
  models/gemini-2.5-pro
  models/gemini-2.0-flash
  models/gemini-2.0-flash-001
  models/gemini-2.0-flash-lite-001
  models/gemini-2.0-flash-lite
  models/gemini-2.5-flash-preview-tts
  models/gemini-2.5-pro-preview-tts
  models/gemma-4-26b-a4b-it
  models/gemma-4-31b-it
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-pro-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-image
  models/gemini-3-pro-preview
  models/gemini-3-flash-preview
  models/gemini-3.1-pro-preview
  models/gemini-3.1-pro-preview-customtools
  models/gemini-3.1-flash-lite-preview
  models/gemini-3.1-flash-lite
  models/gemini-3-pro-image-preview
  models/gemini-3-pro-image
  models/nano-banana-pro-preview
  models/gemini-3.1-flash-image-preview
  models/gemini-3.1-flash-image
  models/gemini-3.1-flash-lite-image
  models/gemini-3.5-flash
  models/gemini-3.5-flash-lite
  models/gemini-omni-flash-preview
  models/gemini-3.6-flash
  models/lyria-3-

set the model and make a basic call

In [ ]:
MODEL = "gemini-3.6-flash"

resp = client.models.generate_content(
    model=MODEL,
    contents="In one sentence, what makes a movie recommendation 'verifiable' rather than just an opinion?",
)
print(resp.text)

A movie recommendation is verifiable when it relies on objective, fact-checkable criteria—such as a specific director, cast, award, or defined plot element—that can be proven to match the user's explicit constraints, rather than subjective assertions of quality.


structured output

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class MoviePlan(BaseModel):
    anchors: list[str]       = Field(description="movie titles the user referenced as anchors, if any")
    mood: str                = Field(description="the mood/vibe the user wants, in a few words")
    runtime_max: Optional[int] = Field(description="max runtime in minutes, or null if none")
    prefer_popular: bool     = Field(description="true if user wants well-known films, false for hidden gems")

resp = client.models.generate_content(
    model=MODEL,
    contents="I want something like Inception but less confusing, under 2 hours, nothing too obscure.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=MoviePlan,        # force output to match this schema
    ),
)
plan = resp.parsed                        # already a validated MoviePlan instance
print(type(plan)); print(plan)

<class '__main__.MoviePlan'>
anchors=['Inception'] mood='accessible sci-fi thriller' runtime_max=120 prefer_popular=True


Write the reusable LLM wrapper ( used by all four agents )


In [ ]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/llm/gemini_client.py
"""Reusable Gemini wrapper for all agents: text + structured (Pydantic) output, with retry."""
import time
from google import genai
from google.genai import types


class GeminiLLM:
    def __init__(self, api_key, model="gemini-3.6-flash", max_retries=5):
        self.client = genai.Client(api_key=api_key)
        self.model = model
        self.max_retries = max_retries

    def _call(self, contents, config=None):
        for attempt in range(self.max_retries):
            try:
                return self.client.models.generate_content(
                    model=self.model, contents=contents, config=config)
            except Exception as e:
                if "429" in str(e) and attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                raise
        raise RuntimeError("exhausted retries")

    def complete(self, prompt, system=None, temperature=0.0):
        cfg = types.GenerateContentConfig(system_instruction=system, temperature=temperature)
        return self._call(prompt, cfg).text

    def structured(self, prompt, schema, system=None, temperature=0.0):
        cfg = types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=schema,
            system_instruction=system,
            temperature=temperature,
        )
        return self._call(prompt, cfg).parsed

Overwriting /content/drive/MyDrive/Projects/multi-agent-discovery/src/llm/gemini_client.py


test the wrapper

In [ ]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, 'src'))
sys.modules.pop('llm.gemini_client', None)
from llm.gemini_client import GeminiLLM

llm = GeminiLLM(api_key=userdata.get('GOOGLE_API_KEY'), model=MODEL)
print(llm.complete("Reply with exactly: wrapper works"))
print(llm.structured("I want a scary hidden-gem horror under 90 minutes.", MoviePlan))

wrapper works
anchors=[] mood='scary horror' runtime_max=90 prefer_popular=False
